# 🤖 Notebook 04 — Model Training
## Bagian 4: Training Model ML + Experiment Runner

**Subset Ringan:**
- Feature Extractors: Glove, FastText, Word2Vec
- Models: Decision Tree, Random Forest, XGBoost

**Total kombinasi:** 9 kombinasi

## Setup

In [7]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import scipy.sparse as sp

from src.experiment_runner import run_experiments

## Automated Experiment Runner

Jalankan kombinasi model ringan dengan Word2Vec.

In [8]:
# Load dataset yang sudah dilabeli dan dipreprocessing
df = pd.read_csv('../data/processed/reviews_prepared.csv')
print(f"📊 Dataset: {df.shape[0]} baris")

# Pastikan kolom yang diperlukan ada
assert 'review_clean' in df.columns, "❌ Jalankan Notebook 02 terlebih dahulu!"
assert 'sentiment_encoded' in df.columns, "❌ Jalankan Notebook 02 terlebih dahulu!"

📊 Dataset: 609 baris


In [9]:
# Jalankan experiment runner (Menggunakan dataset balanced 2024-2026 + TF-IDF, Word2Vec, FastText)
print("\n🚀 Menjalankan Experiment Runner (Balanced Dataset 2024-2026)...")
print("   Extractors: TF-IDF, Word2Vec, FastText")
print("   Models: Decision Tree, Random Forest, XGBoost")
print("   ⏱️  Estimasi waktu: 3-5 menit\n")
print("   ℹ️  Note: Sparse matrix (TF-IDF) akan dikonversi ke dense otomatis")
print("   untuk menghindari sklearn cross-validation ambiguous length error\n")

# Gunakan data yang sudah diseimbangkan dan di-filter tahun 2024-2026
df_balanced = pd.read_csv('../data/processed/reviews_prepared.csv')

print(f"📊 Dataset Info:")
print(f"   Total baris: {len(df_balanced)}")
print(f"   Sentimen distribution: {df_balanced['sentiment'].value_counts().to_dict()}\n")

comparison_df = run_experiments(
    df_balanced,
    text_col='review_clean', 
    label_col='sentiment_encoded',
    subset='priority',
    test_size=0.2,
    random_state=42,
    n_iter=5,
    cv=3,                   
    save_features=True,
    save_dir='../results'
)


🚀 Menjalankan Experiment Runner (Balanced Dataset 2024-2026)...
   Extractors: TF-IDF, Word2Vec, FastText
   Models: Decision Tree, Random Forest, XGBoost
   ⏱️  Estimasi waktu: 3-5 menit

   ℹ️  Note: Sparse matrix (TF-IDF) akan dikonversi ke dense otomatis
   untuk menghindari sklearn cross-validation ambiguous length error

📊 Dataset Info:
   Total baris: 609
   Sentimen distribution: {'Positif': 203, 'Netral': 203, 'Negatif': 203}


🚀 EXPERIMENT RUNNER — Klasifikasi Sentimen CoreTax

📊 Dataset split:
   Train: 487 | Test: 122
   Train distribution: [163 162 162]
   Test distribution:  [40 41 41]

🔧 Extractors: ['GloVe', 'FastText', 'Word2Vec', 'IndoBERT']
🤖 Models: ['Decision Tree', 'Random Forest', 'XGBoost']

📈 Total kombinasi valid: 12
----------------------------------------------------------------------

📦 Feature Extractor: GloVe
   GloVe: Loading vectors dari ../data/embeddings/cc.id.300.vec...
   GloVe: vocab loaded=200000
   GloVe: OOV rate = 183/1372 (13.3%)
   GloVe: sh

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      Batch 1/16
      Batch 11/16
      Batch 1/4
   BERT: shape=(487, 768)
   💾 Features saved: indobert
   ⏱️  Extraction time: 30.7s

   [10/12] 🤖 IndoBERT + Decision Tree
      ✅ W-F1=0.5687 | M-F1=0.5688 | Time=0.3s

   [11/12] 🤖 IndoBERT + Random Forest
      ✅ W-F1=0.6300 | M-F1=0.6294 | Time=4.4s

   [12/12] 🤖 IndoBERT + XGBoost
      ✅ W-F1=0.6302 | M-F1=0.6298 | Time=212.8s


💾 Comparison table saved: ../results/comparison_table.csv

🏆 Top 10 Kombinasi Terbaik (Weighted F1):
Rank |  Feature Extractor |                  Model |   W-F1 |   M-F1 |    AUC | Train(s)
------------------------------------------------------------------------------------------
   1 |              GloVe |          Random Forest | 0.7167 | 0.7164 | 0.8645 |      3.2
   2 |              GloVe |                XGBoost | 0.6911 | 0.6911 | 0.8568 |    103.7
   3 |           Word2Vec |                XGBoost | 0.6392 | 0.6394 | 0.8051 |     23.4
   4 |           IndoBERT |                XGBoost | 0.6302 | 

In [10]:
# Tampilkan hasil
if comparison_df is not None:
    print("\n📊 Tabel Komparasi Lengkap:")
    print(comparison_df[['feature_extractor', 'model', 'weighted_f1', 'macro_f1',
                          'accuracy', 'train_time_s']].to_string())


📊 Tabel Komparasi Lengkap:
     feature_extractor          model  weighted_f1  macro_f1  accuracy  train_time_s
Rank                                                                                
1                GloVe  Random Forest     0.716653  0.716389  0.713115          3.25
2                GloVe        XGBoost     0.691145  0.691083  0.688525        103.67
3             Word2Vec        XGBoost     0.639158  0.639443  0.639344         23.36
4             IndoBERT        XGBoost     0.630176  0.629787  0.631148        212.81
5             IndoBERT  Random Forest     0.629974  0.629380  0.631148          4.43
6             Word2Vec  Random Forest     0.618077  0.618413  0.622951          2.51
7                GloVe  Decision Tree     0.575774  0.575384  0.573770          0.27
8             Word2Vec  Decision Tree     0.573755  0.573657  0.573770          0.05
9             IndoBERT  Decision Tree     0.568661  0.568810  0.565574          0.30
10            FastText        XGBoost

## Ringkasan Model Training

In [11]:
if comparison_df is not None and len(comparison_df) > 0:
    print("\n🏆 Top 3 Kombinasi Terbaik:")
    print("=" * 80)
    top3 = comparison_df.head(3)
    for idx, row in top3.iterrows():
        print(f"   #{idx+1}: {row['feature_extractor']} + {row['model']}")
        print(f"         W-F1={row['weighted_f1']:.4f} | M-F1={row['macro_f1']:.4f} | "
              f"Train={row['train_time_s']:.1f}s")

    best = comparison_df.iloc[0]
    print(f"\n🥇 BEST: {best['feature_extractor']} + {best['model']}")
    print(f"   Weighted F1 = {best['weighted_f1']:.4f}")


🏆 Top 3 Kombinasi Terbaik:
   #2: GloVe + Random Forest
         W-F1=0.7167 | M-F1=0.7164 | Train=3.2s
   #3: GloVe + XGBoost
         W-F1=0.6911 | M-F1=0.6911 | Train=103.7s
   #4: Word2Vec + XGBoost
         W-F1=0.6392 | M-F1=0.6394 | Train=23.4s

🥇 BEST: GloVe + Random Forest
   Weighted F1 = 0.7167


In [12]:
print("\n✅ Model training selesai! Lanjut ke Notebook 05 untuk analisis komparatif.")


✅ Model training selesai! Lanjut ke Notebook 05 untuk analisis komparatif.
